# Mutation sensitivity and interface interpretability

**Status:** evaluation scaffold for review.

**When to run:** after a residue-level model has been trained and frozen.

**Inputs:** a model-specific scorer, wild-type PPI pairs, residue positions,
and independently sourced interface annotations. Optional experimental
mutation labels enable direction/impact validation.

**Outputs:** mutation score deltas, interface-enrichment statistics, and
residue plots under the drafting output folders when enabled.

**Usage rule:** attention weights are hypotheses, not explanations by
themselves. Evaluate enrichment against held-out interface annotations and
compare score changes with experimental mutations where available.

## 1. Setup and file contracts

In [ ]:
from pathlib import Path
import os
import sys

# Keep the repository path separate from SNOOPPI_ROOT, which stores data/artifacts.
# In Colab the recommended clone location is /content/SNOOPPI.
DRAFT_DIR = Path.cwd()
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
    candidate = repository_root / "drafting_code"
    if candidate.exists():
        DRAFT_DIR = candidate
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    raise FileNotFoundError(
        "Cannot locate drafting_code/snooppi_utils.py. Open the notebook from the "
        "repository clone or set SNOOPPI_REPO_ROOT to the clone root."
    )
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL,
    SEQUENCE_A,
    SEQUENCE_B,
    canonical_pair_id,
    ensure_output_directories,
    experiment_paths,
    load_splits,
    normalize_sequence,
    sequence_id,
    split_manifest,
)

PATHS = experiment_paths()
ensure_output_directories(PATHS)
print("SNOOPPI root:", PATHS["root"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import mannwhitneyu, spearmanr

WRITE_OUTPUTS = False
RUN_TAG = "mutation_interface_v1"
MUTATION_TABLE_PATH = PATHS["root"] / "raw" / "mutation_benchmark.csv"
INTERFACE_TABLE_PATH = PATHS["root"] / "raw" / "interface_annotations.csv"

MUTATION_COLUMNS = {
    "pair_id", "partner", "position_1based", "wild_type", "mutant",
    "sequence_A", "sequence_B",
}
INTERFACE_COLUMNS = {"pair_id", "partner", "position_1based", "is_interface"}

print("Mutation input:", MUTATION_TABLE_PATH)
print("Interface input:", INTERFACE_TABLE_PATH)

## 2. Validate annotations and construct mutants

In [ ]:
if not MUTATION_TABLE_PATH.exists():
    raise FileNotFoundError(MUTATION_TABLE_PATH)
mutations = pd.read_csv(MUTATION_TABLE_PATH)
missing = MUTATION_COLUMNS - set(mutations)
if missing:
    raise ValueError(f"Mutation table missing: {sorted(missing)}")

def apply_substitution(sequence, position_1based, wild_type, mutant):
    sequence = str(sequence).upper().replace(" ", "")
    index = int(position_1based) - 1
    if index < 0 or index >= len(sequence):
        raise IndexError(f"Position {position_1based} outside sequence length {len(sequence)}")
    if sequence[index] != wild_type:
        raise ValueError(
            f"Annotation expects {wild_type} at {position_1based}, found {sequence[index]}"
        )
    return sequence[:index] + mutant + sequence[index + 1:]

mutant_rows = []
for row in mutations.itertuples(index=False):
    sequence_a, sequence_b = row.sequence_A, row.sequence_B
    if row.partner == "A":
        mutant_a = apply_substitution(sequence_a, row.position_1based, row.wild_type, row.mutant)
        mutant_b = sequence_b
    elif row.partner == "B":
        mutant_a = sequence_a
        mutant_b = apply_substitution(sequence_b, row.position_1based, row.wild_type, row.mutant)
    else:
        raise ValueError("partner must be A or B")
    mutant_rows.append({**row._asdict(), "mutant_sequence_A": mutant_a, "mutant_sequence_B": mutant_b})
mutant_pairs = pd.DataFrame(mutant_rows)
display(mutant_pairs.head())

## 3. Connect a frozen scorer

Implement `score_pairs` as a thin adapter around the selected checkpoint.
It must return one interaction probability per `(sequence_A, sequence_B)`
pair and must not update model weights. Keeping the adapter explicit avoids
mixing model-specific tokenization into the evaluation logic.

In [ ]:
def score_pairs(sequence_pairs):
    """Replace this body with a frozen-model inference adapter.

    Expected input: list of (sequence_A, sequence_B) strings.
    Expected output: one-dimensional NumPy array in [0, 1].
    """
    raise NotImplementedError("Connect the frozen model checkpoint before running mutation scoring")


RUN_SCORING = False
if RUN_SCORING:
    wild_type_pairs = list(zip(mutant_pairs["sequence_A"], mutant_pairs["sequence_B"]))
    altered_pairs = list(zip(mutant_pairs["mutant_sequence_A"], mutant_pairs["mutant_sequence_B"]))
    mutant_pairs["wild_type_score"] = score_pairs(wild_type_pairs)
    mutant_pairs["mutant_score"] = score_pairs(altered_pairs)
    mutant_pairs["delta_score"] = mutant_pairs["mutant_score"] - mutant_pairs["wild_type_score"]
    if not mutant_pairs[["wild_type_score", "mutant_score"]].apply(lambda column: column.between(0, 1).all()).all():
        raise ValueError("Scorer returned values outside [0, 1]")
    display(mutant_pairs.sort_values("delta_score").head(20))
else:
    print("Scoring disabled. Implement score_pairs, verify the checkpoint, then set RUN_SCORING=True.")

## 4. Compare sensitivity with interface annotations

In [ ]:
if RUN_SCORING:
    if not INTERFACE_TABLE_PATH.exists():
        raise FileNotFoundError(INTERFACE_TABLE_PATH)
    interfaces = pd.read_csv(INTERFACE_TABLE_PATH)
    missing = INTERFACE_COLUMNS - set(interfaces)
    if missing:
        raise ValueError(f"Interface table missing: {sorted(missing)}")
    evaluation = mutant_pairs.merge(
        interfaces,
        on=["pair_id", "partner", "position_1based"],
        how="inner",
        validate="many_to_one",
    )
    evaluation["absolute_delta"] = evaluation["delta_score"].abs()
    interface_delta = evaluation.loc[evaluation["is_interface"].eq(1), "absolute_delta"]
    noninterface_delta = evaluation.loc[evaluation["is_interface"].eq(0), "absolute_delta"]
    if interface_delta.empty or noninterface_delta.empty:
        raise ValueError("Both interface and non-interface residues are required")
    statistic, p_value = mannwhitneyu(interface_delta, noninterface_delta, alternative="greater")
    enrichment_summary = pd.DataFrame([{
        "interface_mutations": len(interface_delta),
        "noninterface_mutations": len(noninterface_delta),
        "interface_median_absolute_delta": interface_delta.median(),
        "noninterface_median_absolute_delta": noninterface_delta.median(),
        "mannwhitney_u": statistic,
        "one_sided_p_value": p_value,
    }])
    display(enrichment_summary.round(4))

    if "experimental_effect" in evaluation:
        correlation, correlation_p = spearmanr(evaluation["delta_score"], evaluation["experimental_effect"], nan_policy="omit")
        print("Spearman model-vs-experimental effect:", correlation, "p=", correlation_p)

## 5. Residue-level visualization and optional outputs

In [ ]:
if RUN_SCORING:
    selected_pair = evaluation["pair_id"].iloc[0]
    selected = evaluation.loc[evaluation["pair_id"].eq(selected_pair)].sort_values("position_1based")
    figure, axis = plt.subplots(figsize=(10, 4.5))
    colors = np.where(selected["is_interface"].eq(1), "#D58632", "#176B87")
    axis.scatter(selected["position_1based"], selected["delta_score"], c=colors, alpha=0.8)
    axis.axhline(0, color="#333333", linewidth=1)
    axis.set(
        title=f"In-silico mutation effects for pair {selected_pair}",
        xlabel="Residue position (1-based)",
        ylabel="Mutant − wild-type interaction score",
    )
    figure.tight_layout()
    plt.show()

    if WRITE_OUTPUTS:
        table_path = PATHS["tables"] / f"{RUN_TAG}_mutation_scores.csv"
        figure_path = PATHS["figures"] / f"{RUN_TAG}_{selected_pair}_mutation_scan.png"
        if table_path.exists() or figure_path.exists():
            raise FileExistsError("Change RUN_TAG; refusing to overwrite interpretability outputs")
        evaluation.to_csv(table_path, index=False)
        figure.savefig(figure_path, dpi=300, bbox_inches="tight")
else:
    print("No figures or outputs until RUN_SCORING=True.")

## Interpretation limits

- Use annotations not seen in training or model selection.
- Correct for repeated mutations from the same protein/pair in uncertainty
  estimates; pair-level bootstrap is preferable to treating residues as
  independent.
- A predicted score delta is not a binding free-energy estimate.
- Prioritize a small, preregistered mutation set for literature or wet-lab
  validation rather than selecting only visually convincing examples.